# Employment stability analysis

Fifth package. Sources: `LastJobInfo.Job` (employer, tenure `Duration`, first/last
payment), `Income.workDependent` (SS income records: qualification TCO/MOE, paying
entities), `EndOfWorkContractsList` (contract terminations), and the NetSalary series
(continuity).

- **Layer 1** `employment_features(profile)`: versioned vector.
- **Layer 2** `display_profile` / `render_profile_text` (pt-PT).

Design decisions:
1. Employment data arrives through the **Social Security gate** (completeness
   contract): absence usually means no data access, not unemployment.
2. Main job = the record with the longest `Duration`.
3. `salary_growth_over_tenure` is computed only between **full months**
   (`Days >= 28` on both payments): a partial admission month would fake
   spectacular growth (observed: +347% from a 6-day first month).
4. Month = 0 aggregate rows are excluded from the continuity series.

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from employment_features import employment_features, render_profile_text

pd.set_option("display.width", 200)
raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
feats = pd.DataFrame([
    {"uuid": u, **employment_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"{feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

## 1. Population overview

In [ ]:
print("with employment data:", int(feats["has_employment_data"].sum()),
      "of", len(feats), "(gated by SS access)")
KEY = ["tenure_months", "salary_growth_over_tenure", "salary_streak_months",
       "salary_gap_months", "n_terminations", "months_since_last_termination",
       "n_paying_entities"]
print(feats[KEY].describe(percentiles=[.25, .5, .75, .9]).round(2).T.to_string())

print("\nrisk-relevant states:")
print("  employer company NOT active:", int((feats["employer_active"] == 0).sum()))
print("  termination in last 24 months:", int((feats["terminations_last_24m"].fillna(0) > 0).sum()))
print("  salary series with holes:", int((feats["salary_gap_months"].fillna(0) > 0).sum()))
print("  board members (MOE):", int((feats["is_board_member"] == 1).sum()))
print("  3+ paying entities:", int((feats["n_paying_entities"].fillna(0) >= 3).sum()))

## 2. Layer 2 examples: stable, fragile, and no-data

In [ ]:
fset = feats.set_index("uuid")
vmap = dict(zip(snap["uuid"], snap["value"]))
examples = [
    fset[(fset["tenure_months"] > 60) & (fset["employer_active"] == 1)
         & (fset["n_terminations"].isna())].index[0],
    fset[(fset["employer_active"] == 0)
         & (fset["terminations_last_24m"].fillna(0) > 0)].index[0],
    fset[fset["has_employment_data"] == 0].index[0],
]
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(employment_features(json.loads(vmap[u]))))
    print()

## 3. Validation: module vs independent recomputation. Must print 0.

In [ ]:
def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vec = employment_features(p)
    jobs = dig(p, "LastJobInfo", "Job") or []
    expect = {}
    if jobs:
        main = max(jobs, key=lambda j: j.get("Duration") or 0)
        expect["tenure_months"] = main.get("Duration")
        fp, lp = main.get("FirstPayment") or {}, main.get("LastPayment") or {}
        if (fp.get("Amount") and lp.get("Amount") and fp["Amount"] > 0
                and fp.get("Days", 0) >= 28 and lp.get("Days", 0) >= 28):
            expect["salary_growth_over_tenure"] = round(lp["Amount"] / fp["Amount"] - 1, 4)
        else:
            expect["salary_growth_over_tenure"] = None
    terms = [t for t in (p.get("EndOfWorkContractsList") or []) if t.get("EndDate")]
    expect["n_terminations"] = len(terms) or None
    sal = sorted({int(e["Year"]) * 12 + int(e["Month"]) - 1
                  for e in (p.get("NetSalary") or [])
                  if e.get("Year") and e.get("Month")})
    if sal:
        streak = 1
        for a, b in zip(reversed(sal[:-1]), reversed(sal[1:])):
            if b - a == 1:
                streak += 1
            else:
                break
        expect["salary_streak_months"] = streak
        expect["salary_gap_months"] = (sal[-1] - sal[0] + 1 - len(sal)
                                       if len(sal) > 1 else 0)
    for k, e in expect.items():
        checked += 1
        if vec[k] != e:
            mism += 1
            print("MISMATCH", uuid[:8], k, vec[k], e)
print(f"{checked} comparisons, mismatches: {mism}")